# Etapa 2 — CNN do zero com PyTorch e TorchVision

Este notebook é o artefato de entrega da segunda etapa: uma CNN criada camada a camada para classificar imagens de gatos e cães. Não há arquitetura pronta nem pesos pré-treinados escondidos atrás da cortina, porque a etapa pede justamente isso.

**Conformidade principal**

- entrada RGB de `3 × 224 × 224`;
- seis camadas convolucionais, três `MaxPool2d` e camadas totalmente conectadas;
- treinamento com `BCEWithLogitsLoss` e `AdamW`;
- quatro técnicas de augmentation: crop/escala, flip horizontal, color jitter e random erasing;
- métricas de loss, acurácia, precision, recall, F1, matriz de confusão e tempo de treinamento.


## 1. Estrutura do dataset

A divisão `train/val/test` é a fornecida pelo professor e não deve ser recriada pelo notebook.

```text
data/
├── train/{cats,dogs}/
├── val/{cats,dogs}/
└── test/{cats,dogs}/
```

Caso as pastas sejam `gatos/cachorros`, ajuste apenas `POSITIVE_CLASS` abaixo. O projeto aceita qualquer par de nomes, desde que os três splits tenham a mesma taxonomia.


In [ ]:
from pathlib import Path
import sys

# Executar o notebook a partir da raiz do repositório ou de notebooks/.
ROOT = Path.cwd().resolve()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd
import matplotlib.pyplot as plt
import torch

from cnn_cats_dogs.config import TrainingConfig
from cnn_cats_dogs.data import RGB_MEAN, RGB_STD, build_data_bundle
from cnn_cats_dogs.engine import run_training
from cnn_cats_dogs.model import ScratchCNN, count_trainable_parameters

print(f'Raiz do projeto: {ROOT}')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA disponível: {torch.cuda.is_available()}')


## 2. Hiperparâmetros e reprodutibilidade

A seed fixa permite repetir o experimento. A seleção do melhor modelo é feita pela **loss de validação**. O conjunto de teste é avaliado apenas depois do carregamento do melhor checkpoint.


In [ ]:
DATA_DIR = ROOT / 'data'                 # ajuste se o dataset estiver em outro local
OUTPUT_DIR = ROOT / 'runs' / 'cnn_scratch'
POSITIVE_CLASS = 'dogs'                   # por exemplo: 'cachorros' se essa for a pasta positiva

config = TrainingConfig(
    data_dir=DATA_DIR,
    output_dir=OUTPUT_DIR,
    positive_class=POSITIVE_CLASS,
    image_size=224,
    batch_size=32,
    epochs=25,
    learning_rate=1e-3,
    weight_decay=1e-4,
    num_workers=0,                         # Windows: deixe 0; Linux/Colab: 2 ou 4 costuma funcionar
    seed=42,
    device='auto',
    use_amp=True,
    early_stopping_patience=8,
)
config.validate()
config


## 3. Data augmentation

As transformações de treino pertencem a grupos distintos:

1. **crop/escala espacial**: `RandomResizedCrop`;
2. **geometria por reflexão**: `RandomHorizontalFlip`;
3. **fotométrica**: `ColorJitter`;
4. **oclusão/regularização**: `RandomErasing`.

Validação e teste usam apenas preprocessamento determinístico. Isso evita o equivalente estatístico de mudar a prova a cada correção.


In [ ]:
bundle = build_data_bundle(config)
print('Classes [0, 1]:', bundle.class_names)
print('Tamanhos:', bundle.sizes)

images, labels = next(iter(bundle.train_loader))

def denormalize(image):
    mean = torch.tensor(RGB_MEAN).view(3, 1, 1)
    std = torch.tensor(RGB_STD).view(3, 1, 1)
    return (image.cpu() * std + mean).clamp(0, 1)

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, image, label in zip(axes.flat, images[:8], labels[:8]):
    ax.imshow(denormalize(image).permute(1, 2, 0))
    ax.set_title(bundle.class_names[int(label)])
    ax.axis('off')
fig.suptitle('Amostras com augmentation de treino')
fig.tight_layout()


## 4. Arquitetura da CNN

A rede abaixo é definida diretamente no projeto. Ela recebe RGB `224×224`, extrai atributos por blocos convolucionais e transforma a representação final em **um logit**. A sigmoide é aplicada apenas para gerar probabilidades durante as métricas, enquanto a loss recebe logits diretamente por estabilidade numérica.


In [ ]:
model_preview = ScratchCNN()
print(model_preview)
print(f'Parâmetros treináveis: {count_trainable_parameters(model_preview):,}')


## 5. Treinamento

O código salva automaticamente configurações, ambiente de execução, histórico por época, checkpoints, curvas de aprendizado, matriz de confusão e predições do teste em `runs/cnn_scratch/`. Isso deixa o relatório menos dependente de memória humana, uma tecnologia instável.


In [ ]:
artifacts = run_training(config)
artifacts


## 6. Resultados no conjunto de teste

O bloco seguinte abre os arquivos gerados. Depois da execução, use estes valores e gráficos diretamente na seção de resultados do relatório.


In [ ]:
import json

with artifacts.summary_path.open(encoding='utf-8') as file:
    summary = json.load(file)

print('Melhor época:', summary['best_epoch'])
print('Tempo total (s):', round(float(summary['training_time_seconds']), 2))
print('Métricas de teste:')
pd.Series(summary['test_metrics'])


In [ ]:
from IPython.display import Image, display

display(Image(filename=str(OUTPUT_DIR / 'plots' / 'learning_curves.png')))
display(Image(filename=str(OUTPUT_DIR / 'plots' / 'confusion_matrix_test.png')))

history = pd.read_csv(OUTPUT_DIR / 'artifacts' / 'history.csv')
history.tail()


## 7. Pontos para discussão no relatório

- Compare a diferença entre treino e validação para avaliar sobreajuste.
- Discuta o efeito das augmentations sobre a generalização.
- Explique a matriz de confusão, destacando falsos positivos e falsos negativos.
- Registre hardware, device usado, batch size e tempo total a partir de `experiment_config.json`.
- Para a Etapa 3, mantenha a mesma divisão e o mesmo formato de artefatos. O plano de transição está em `docs/part3_handoff.md`.
